<a href="https://colab.research.google.com/github/comsadavid/AI-solutions/blob/main/competitions/mlcompete/famous-paintings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import adjusted_rand_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from catboost import CatBoostRegressor
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

In [ ]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

In [ ]:
df_train.head()

,SampleID,canvas_size,is_oil_painting,brush_type,num_colors,colorfulness,complexity,brightness,contrast,stroke_density,has_signature,is_framed,uses_gold_leaf,is_restored,dominant_warm_colors,dominant_color,art_period_hint,auction_house,image_quality,brightness_log,complexity_x_stroke,fake_style_score,painter_style_score,target_price
0,332,60x50,True,medium,71,0.616240,0.755582,0.647338,0.587923,0.702118,True,True,False,True,False,red,baroque,Online,low,0.499161,0.530508,0.349718,0.536437,50800
1,410,80x90,True,medium,60,0.660715,0.474923,0.538822,0.599076,0.528112,True,False,False,True,False,yellow,modern,Online,low,0.431017,0.250813,0.258722,0.163906,37400
2,77,80x50,True,fine,64,0.684877,0.380591,0.608029,0.500152,0.508521,True,False,False,False,False,mixed,baroque,Sothebys,low,0.475009,0.193539,0.797662,0.137732,35500
3,869,80x50,True,medium,56,0.427938,0.581636,0.562086,0.483896,0.550152,True,False,False,True,False,blue,modern,Local,low,0.446022,0.319988,0.569981,0.394542,43200
4,139,80x130,True,fine,55,0.481406,0.629780,0.476093,0.493429,0.681710,True,True,False,False,False,red,modern,Local,medium,0.389398,0.429328,0.723147,0.321858,54500


In [ ]:
def saa(df_test):
    s=0
    if df_test['stroke_density'] > 0.7: s+=2
    if df_test['complexity'] > 0.65: s+=2
    if df_test['uses_gold_leaf'] == 1: s+=1
    if df_test['has_signature'] == 1: s+=1
    if df_test['num_colors'] > 65 and df_test['colorfulness'] > 0.7: s+=2
    if df_test['contrast'] < 0.4 or df_test['brightness'] < 0.45 or df_test['brightness'] > 0.75: s-=1;
    return 'Autentic' if s>=5 else 'Incert'
output1 = pd.DataFrame({'SampleID': df_test['SampleID'], 'subtaskID': 'Task1', 'Answer': df_test.apply(saa, axis=1)})

In [ ]:
df_train_copy = df_train[['painter_style_score']]
df_test_copy = df_test.copy()

In [ ]:
sc = StandardScaler()
X_scaled = sc.fit_transform(df_train_copy)

In [ ]:
model = KMeans(n_clusters=5, random_state=42)
model.fit(X_scaled)

KMeans(n_clusters=5, random_state=42)

In [ ]:
def solve_k(df_test_copy):
    X = df_test_copy[['painter_style_score']]
    X_scaled = sc.transform(X)
    output2 = pd.DataFrame({'SampleID': df_test_copy['SampleID'], 'subtaskID': 'Task2', 'Answer': model.predict(X_scaled).astype(int)})
    return output2

In [ ]:
output2 = solve_k(df_test_copy)

In [ ]:
# Predict clusters for the training data
X_train_clusters = model.predict(X_scaled) # X_scaled is df_train's scaled painter_style_score
df_train['painter'] = X_train_clusters

# Predict clusters for the test data
X_test_scaled = sc.transform(df_test_copy[['painter_style_score']])
df_test['painter'] = model.predict(X_test_scaled)

In [ ]:
 def encode_columns(df_train, df_test, columns):
    ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore', dtype=int)
    encoded_train_list = ohe.fit_transform(df_train[columns])
    encoded_test_list = ohe.transform(df_test[columns])

    encoded_df_train = pd.DataFrame(encoded_train_list, columns=ohe.get_feature_names_out())
    encoded_df_test = pd.DataFrame(encoded_test_list, columns=ohe.get_feature_names_out())

    df_train[encoded_df_train.columns] = encoded_df_train
    df_test[encoded_df_test.columns] = encoded_df_test

    df_train.drop(columns, axis=1, inplace=True)
    df_test.drop(columns, axis=1, inplace=True)

def proces(df_train, df_test):
    encode_list = ['brush_type', 'dominant_color', 'art_period_hint', 'auction_house', 'image_quality', 'painter']
    encode_columns(df_train, df_test, encode_list)

    drop_list = ['fake_style_score']
    df_train.drop(drop_list, axis=1, inplace=True)
    df_test.drop(drop_list, axis=1, inplace=True)

    df_train['canvas_size'] = df_train['canvas_size'].str.split("x").str[0].astype(float) * df_train['canvas_size'].str.split("x").str[1].astype(float)
    df_test['canvas_size'] = df_test['canvas_size'].str.split("x").str[0].astype(float) * df_test['canvas_size'].str.split("x").str[1].astype(float)


In [ ]:
proces(df_train, df_test)

In [ ]:
X = df_train.drop(['SampleID','target_price'], axis=1)
y = df_train['target_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
mms = MinMaxScaler()
X_train = mms.fit_transform(X_train)
X_test = mms.transform(X_test)

In [ ]:
def evaluate_model(model):
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    print(f"Train: {mean_absolute_error(y_train, y_train_pred)}, Test: {mean_absolute_error(y_test, y_test_pred)}")

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
evaluate_model(lr)

Train: 379.04547613140363, Test: 401.68906561940304


In [ ]:
ct = CatBoostRegressor(verbose=0)
ct.fit(X_train, y_train)
evaluate_model(ct)

Train: 138.76675002545156, Test: 714.8895427499953


In [ ]:
def solve(model):

    X_pred = df_test.drop(['SampleID'], axis=1)
    X_pred = mms.transform(X_pred)
    y_pred = model.predict(X_pred)

    output3 = pd.DataFrame({'SampleID': df_test['SampleID'], 'subtaskID': 'Task3', 'Answer': y_pred.astype(int)})

    output = pd.concat([output1, output2, output3])
    output.to_csv('output.csv', index=False)

In [ ]:
solve(lr)